In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, make_scorer
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

def regularization_analysis_model3():
    """
    Análise de regularização para Modelo 3 (dataset combinado)
    usando a mesma metodologia rigorosa do Modelo 1:
    - Nested cross-validation
    - Grid search
    - RobustScaler
    - Class balancing
    - IC 95%
    """
    
    # Carregar dados de treinamento do Modelo 3
    train_path = '/Users/marcelosilva/Desktop/early-obesity-prediction/D-Train-Test Split/MODEL3TRAIN.csv'
    df = pd.read_csv(train_path)
    
    print("="*80)
    print("ANÁLISE COMPARATIVA DE REGULARIZAÇÃO - MODELO 3")
    print("="*80)
    
    # Preparar dados
    target_cols = ['desnutrido', 'eutrofico', 'sobrepeso', 'obeso']
    y = df['obeso'].copy()
    
    # Features: excluir id_anon e todas as variáveis target
    feature_cols = [col for col in df.columns if col not in target_cols + ['id_anon']]
    X = df[feature_cols].copy()
    
    print(f"Dataset de treinamento Modelo 3:")
    print(f"  - Observações: {len(df):,}")
    print(f"  - Features: {len(feature_cols)}")
    print(f"  - Obesos: {y.sum()} ({y.mean()*100:.1f}%)")
    print(f"  - Não-obesos: {(1-y).sum()} ({(1-y.mean())*100:.1f}%)")
    
    # Categorizar features por origem
    model1_features = []  # Demográficas/perinatais
    model2_features = []  # Alimentação/amamentação
    
    # Keywords para classificação automática
    demographic_keywords = ['regiao_', 'cor_', 'parto_', 'sexo_', 'idade_', 'semanas_', 'peso_nascimento', 
                           'altura_nascimento', 'tipo_de_parto', 'chupeta_', 'sabe_ler', 'sindrome_', 
                           'teve_exposicao', 'nivel_educacional', 'prenatal_consultas', 'recebe_beneficio', 
                           'vd_ien_escore']
    
    feeding_keywords = ['k08_', 'k07_', 'k06_', 'k11_', 'k03_', 'k04_', 'k15_', 'tempo_primeira', 
                       'usou_utensilios', 'dias_aleitamento', 'doou_leite', 'recebeu_leite', 
                       'amamentou_outra', 'usou_mamadeira', 'deixou_amamentar', 'busca_info', 
                       'deu_outros_liquidos']
    
    for feature in feature_cols:
        is_demographic = any(keyword in feature for keyword in demographic_keywords)
        is_feeding = any(keyword in feature for keyword in feeding_keywords)
        
        if is_demographic:
            model1_features.append(feature)
        elif is_feeding:
            model2_features.append(feature)
        else:
            # Classificar manualmente features restantes
            if 'k05_prenatal_consultas' in feature:
                model1_features.append(feature)
            else:
                model2_features.append(feature)
    
    print(f"\nClassificação das features:")
    print(f"  - Origem Modelo 1 (demográficas/perinatais): {len(model1_features)} features")
    print(f"  - Origem Modelo 2 (alimentação/amamentação): {len(model2_features)} features")
    print(f"  - Total combinado: {len(model1_features) + len(model2_features)} features")
    
    # Definir grupos de features para análise (expandido para incluir alimentação)
    feature_groups = {
        'regiao': [f for f in feature_cols if f.startswith('regiao_')],
        'cor_raca': [f for f in feature_cols if f.startswith('cor_')],
        'tipo_parto': [f for f in feature_cols if 'parto_' in f],
        'demograficas': ['sexo_masculino', 'idade_crianca', 'idade_materna_ao_nascimento'],
        'perinatais': ['semanas_gravidez', 'peso_nascimento', 'altura_nascimento', 'tipo_de_parto'],
        'socioeconomicas': ['sabe_ler', 'nivel_educacional_mae', 'recebe_beneficio', 'vd_ien_escore'],
        'peso_materno': [f for f in feature_cols if f.startswith('k0') and ('peso' in f or 'quilos' in f)],
        'prenatal': [f for f in feature_cols if 'prenatal' in f or 'k03_' in f or 'k04_' in f],
        'amamentacao': [f for f in feature_cols if any(kw in f for kw in ['amament', 'leite', 'mama'])],
        'praticas_alimentares': [f for f in feature_cols if any(kw in f for kw in ['liquidos', 'utensilios', 'busca_info'])]
    }
    
    # Identificar features não categorizadas
    all_categorized = []
    for group_features in feature_groups.values():
        all_categorized.extend(group_features)
    
    outras_features = [f for f in feature_cols if f not in all_categorized]
    if outras_features:
        feature_groups['outras'] = outras_features
    
    # Atualizar grupos removendo listas vazias
    feature_groups = {k: v for k, v in feature_groups.items() if v}
    
    print(f"\nGrupos de features identificados:")
    for grupo, features in feature_groups.items():
        available_features = [f for f in features if f in feature_cols]
        if available_features:
            print(f"  - {grupo.replace('_', ' ').title()}: {len(available_features)} variáveis")
    
    # Configurações de cross-validation
    outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    # Parâmetros para grid search (mesmos do Modelo 1)
    param_grids = {
        'lasso': {
            'model__C': [0.1, 1.0, 10.0, 100.0, 1000.0],
            'model__penalty': ['l1']
        },
        'ridge': {
            'model__C': [0.1, 1.0, 10.0, 100.0, 1000.0],
            'model__penalty': ['l2']
        },
        'elastic_net': {
            'model__C': [0.1, 1.0, 10.0, 100.0, 1000.0],
            'model__penalty': ['elasticnet'],
            'model__l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]
        }
    }
    
    def calculate_confidence_intervals(scores, confidence=0.95):
        """Calcula intervalos de confiança usando distribuição t"""
        n = len(scores)
        mean_score = np.mean(scores)
        std_err = stats.sem(scores)
        h = std_err * stats.t.ppf((1 + confidence) / 2., n-1)
        return mean_score, mean_score - h, mean_score + h
    
    def evaluate_regularization_method(X_data, y_data, model, param_grid, method_name):
        """Avalia um método de regularização com nested cross-validation"""
        
        print(f"\n🔍 Avaliando {method_name}...")
        
        # Pipeline com preprocessamento
        pipeline = Pipeline([
            ('scaler', RobustScaler()),
            ('model', model)
        ])
        
        # Nested cross-validation
        results = []
        best_params_list = []
        selected_features_list = []
        
        for fold, (train_idx, val_idx) in enumerate(outer_cv.split(X_data, y_data)):
            X_train_fold, X_val_fold = X_data.iloc[train_idx], X_data.iloc[val_idx]
            y_train_fold, y_val_fold = y_data.iloc[train_idx], y_data.iloc[val_idx]
            
            # Grid search no inner CV
            grid_search = GridSearchCV(
                pipeline, param_grid, cv=inner_cv, 
                scoring='roc_auc', n_jobs=-1, verbose=0
            )
            
            grid_search.fit(X_train_fold, y_train_fold)
            best_params_list.append(grid_search.best_params_)
            
            # Avaliar no validation fold
            y_pred_proba = grid_search.predict_proba(X_val_fold)[:, 1]
            y_pred = grid_search.predict(X_val_fold)
            
            # Calcular métricas
            fold_results = {
                'auc': roc_auc_score(y_val_fold, y_pred_proba),
                'precision': precision_score(y_val_fold, y_pred, zero_division=0),
                'recall': recall_score(y_val_fold, y_pred, zero_division=0),
                'f1': f1_score(y_val_fold, y_pred, zero_division=0)
            }
            results.append(fold_results)
            
            # Features selecionadas (para Lasso e Elastic Net)
            if hasattr(grid_search.best_estimator_.named_steps['model'], 'coef_'):
                coefs = grid_search.best_estimator_.named_steps['model'].coef_
                if coefs.ndim > 1:
                    coefs = coefs.flatten()
                selected_features = X_data.columns[coefs != 0].tolist()
                selected_features_list.append(selected_features)
        
        # Consolidar resultados
        metrics_summary = {}
        for metric in ['auc', 'precision', 'recall', 'f1']:
            scores = [r[metric] for r in results]
            mean_score, ci_lower, ci_upper = calculate_confidence_intervals(scores)
            metrics_summary[metric] = {
                'mean': mean_score,
                'std': np.std(scores),
                'ci_lower': ci_lower,
                'ci_upper': ci_upper,
                'scores': scores
            }
        
        return {
            'method': method_name,
            'metrics': metrics_summary,
            'best_params': best_params_list,
            'selected_features': selected_features_list,
            'n_features_selected': [len(sf) for sf in selected_features_list] if selected_features_list else None
        }
    
    # =======================================================================
    # AVALIAÇÃO DOS MÉTODOS DE REGULARIZAÇÃO
    # =======================================================================
    print(f"\n" + "="*60)
    print("AVALIAÇÃO DE MÉTODOS DE REGULARIZAÇÃO")
    print("="*60)
    
    results = {}
    
    # Lasso
    lasso_results = evaluate_regularization_method(
        X, y, LogisticRegression(random_state=42, max_iter=10000, penalty='l1', solver='liblinear', class_weight='balanced'), 
        param_grids['lasso'], 'Lasso'
    )
    results['lasso'] = lasso_results
    
    # Ridge  
    ridge_results = evaluate_regularization_method(
        X, y, LogisticRegression(random_state=42, max_iter=10000, penalty='l2', class_weight='balanced'), 
        param_grids['ridge'], 'Ridge'
    )
    results['ridge'] = ridge_results
    
    # Elastic Net
    elastic_results = evaluate_regularization_method(
        X, y, LogisticRegression(random_state=42, max_iter=10000, penalty='elasticnet', solver='saga', class_weight='balanced'), 
        param_grids['elastic_net'], 'Elastic Net'
    )
    results['elastic_net'] = elastic_results
    
    # =======================================================================
    # RESULTADOS COMPARATIVOS
    # =======================================================================
    print(f"\n" + "="*80)
    print("RESULTADOS COMPARATIVOS - MODELO 3")
    print("="*80)
    
    print(f"{'Método':<15} {'AUC-ROC':<25} {'Precision':<25} {'Recall':<25} {'F1-Score':<25}")
    print("-" * 115)
    
    for method_key, result in results.items():
        method_name = result['method']
        auc = result['metrics']['auc']
        precision = result['metrics']['precision']
        recall = result['metrics']['recall']
        f1 = result['metrics']['f1']
        
        auc_str = f"{auc['mean']:.3f} ({auc['ci_lower']:.3f}-{auc['ci_upper']:.3f})"
        prec_str = f"{precision['mean']:.3f} ({precision['ci_lower']:.3f}-{precision['ci_upper']:.3f})"
        rec_str = f"{recall['mean']:.3f} ({recall['ci_lower']:.3f}-{recall['ci_upper']:.3f})"
        f1_str = f"{f1['mean']:.3f} ({f1['ci_lower']:.3f}-{f1['ci_upper']:.3f})"
        
        print(f"{method_name:<15} {auc_str:<25} {prec_str:<25} {rec_str:<25} {f1_str:<25}")
    
    # Análise de feature selection
    print(f"\n" + "="*60)
    print("ANÁLISE DE FEATURE SELECTION")
    print("="*60)
    
    for method_key, result in results.items():
        if result['n_features_selected']:
            n_features = result['n_features_selected']
            method_name = result['method']
            print(f"  {method_name}: {np.mean(n_features):.1f} ± {np.std(n_features):.1f} features selecionadas")
    
    # Análise de features por categoria (para o melhor método)
    best_method_key = max(results.keys(), key=lambda k: results[k]['metrics']['auc']['mean'])
    best_result = results[best_method_key]
    
    if best_result['selected_features']:
        print(f"\nAnálise de features selecionadas pelo melhor método ({best_result['method']}):")
        
        # Contar features selecionadas por categoria
        all_selected = []
        for feature_list in best_result['selected_features']:
            all_selected.extend(feature_list)
        
        feature_counts = {}
        for feature in set(all_selected):
            feature_counts[feature] = all_selected.count(feature)
        
        # Agrupar por categoria
        category_selections = {}
        for grupo, group_features in feature_groups.items():
            selected_in_group = [f for f in group_features if f in feature_counts]
            if selected_in_group:
                category_selections[grupo] = selected_in_group
        
        for categoria, features in category_selections.items():
            print(f"  - {categoria.replace('_', ' ').title()}: {len(features)} features")
            for feature in features[:3]:  # Mostrar apenas as 3 primeiras
                frequency = feature_counts[feature]
                print(f"    * {feature} (selecionada em {frequency}/5 folds)")
    
    # Identificar melhor método
    print(f"\n" + "="*60)
    print("RECOMENDAÇÃO METODOLÓGICA")
    print("="*60)
    
    all_results = []
    for method_key, result in results.items():
        all_results.append({
            'method': result['method'],
            'auc_mean': result['metrics']['auc']['mean'],
            'auc_ci_width': result['metrics']['auc']['ci_upper'] - result['metrics']['auc']['ci_lower'],
            'auc_std': result['metrics']['auc']['std']
        })
    
    # Ordenar por AUC médio
    all_results.sort(key=lambda x: x['auc_mean'], reverse=True)
    
    print(f"Ranking por AUC-ROC médio:")
    for i, result in enumerate(all_results, 1):
        print(f"{i}. {result['method']}: AUC = {result['auc_mean']:.3f} ± {result['auc_std']:.3f}")
    
    best_method = all_results[0]
    print(f"\nMÉTODO RECOMENDADO: {best_method['method']}")
    print(f"  - AUC-ROC: {best_method['auc_mean']:.3f}")
    print(f"  - Estabilidade (std): {best_method['auc_std']:.3f}")
    
    # Comparação com modelos anteriores
    print(f"\n" + "="*60)
    print("COMPARAÇÃO COM MODELOS ANTERIORES")
    print("="*60)
    
    print(f"RESULTADOS DE REGULARIZAÇÃO:")
    print(f"Modelo 1 (Demográficas/Perinatais):")
    print(f"  - Melhor: Lasso AUC = 0.543")
    print(f"  - Features: 27 variáveis")
    print(f"  - Amostra: 6,588 observações")
    
    print(f"\nModelo 2 (Práticas Alimentares):")
    print(f"  - Melhor: Ridge AUC = 0.600")
    print(f"  - Features: 17 variáveis") 
    print(f"  - Amostra: 4,510 observações")
    
    print(f"\nModelo 3 (Combinado):")
    print(f"  - Melhor: {best_method['method']} AUC = {best_method['auc_mean']:.3f}")
    print(f"  - Features: {len(feature_cols)} variáveis")
    print(f"  - Amostra: {len(df):,} observações")
    
    # Análise comparativa
    modelo2_auc = 0.600
    modelo1_auc = 0.543
    
    if best_method['auc_mean'] > modelo2_auc:
        diff = best_method['auc_mean'] - modelo2_auc
        print(f"\n✓ Modelo 3 supera Modelo 2 (+{diff:.3f} AUC-ROC)")
    elif best_method['auc_mean'] > modelo1_auc:
        diff = best_method['auc_mean'] - modelo1_auc
        print(f"\n✓ Modelo 3 supera Modelo 1 (+{diff:.3f} AUC-ROC) mas não Modelo 2")
    else:
        print(f"\n⚠ Modelo 3 não supera modelos anteriores")
    
    print(f"\nConclusões preliminares:")
    if best_method['auc_mean'] < 0.65:
        print(f"  - AUC < 0.65 sugere limitações preditivas persistentes")
        print(f"  - Combinação de variáveis não resolve problema fundamental")
        print(f"  - Evidência robusta de que fatores dos primeiros 24 meses têm")
        print(f"    capacidade preditiva limitada para obesidade aos 2-4 anos")
    
    return results, best_method

# Executar análise
if __name__ == "__main__":
    model3_results, best_model3 = regularization_analysis_model3()

ANÁLISE COMPARATIVA DE REGULARIZAÇÃO - MODELO 3
Dataset de treinamento Modelo 3:
  - Observações: 4,510
  - Features: 44
  - Obesos: 134 (3.0%)
  - Não-obesos: 4376 (97.0%)

Classificação das features:
  - Origem Modelo 1 (demográficas/perinatais): 27 features
  - Origem Modelo 2 (alimentação/amamentação): 17 features
  - Total combinado: 44 features

Grupos de features identificados:
  - Regiao: 5 variáveis
  - Cor Raca: 4 variáveis
  - Tipo Parto: 3 variáveis
  - Demograficas: 3 variáveis
  - Perinatais: 4 variáveis
  - Socioeconomicas: 4 variáveis
  - Peso Materno: 3 variáveis
  - Prenatal: 3 variáveis
  - Amamentacao: 8 variáveis
  - Praticas Alimentares: 3 variáveis
  - Outras: 5 variáveis

AVALIAÇÃO DE MÉTODOS DE REGULARIZAÇÃO

🔍 Avaliando Lasso...

🔍 Avaliando Ridge...

🔍 Avaliando Elastic Net...

RESULTADOS COMPARATIVOS - MODELO 3
Método          AUC-ROC                   Precision                 Recall                    F1-Score                 
-----------------------------